# Module 13B - LoRA

Use this notebook after `tests/test_lora.py` is passing and after you have finished Module 13 (the LoRA run reuses your hand-authored instruction dataset). The notebook injects low-rank adapters into BaseLM, verifies the exact no-op at initialization, freezes everything but the adapters, reruns the Module 13 SFT recipe against a fraction of a percent of the parameters, then merges, unmerges, and ships the adapter as a file a thousand times smaller than the model.

The important work is watching what stays the same: the data, the loss, the trainer are all Module 13's. Only the set of parameters allowed to move has changed.

1. Read the lesson page (`docs/modules/13b-lora.md`).
2. Open this notebook with `./notebook.sh 13b`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

## Setup

In [ ]:
from pathlib import Path
import copy
import json
import subprocess
import sys
import time

import torch

from g2c.artifacts import load_model_artifact_with_tokenizer
from g2c.lora import (
    LoRAModel,
    count_parameters,
    inject_lora,
    load_lora_state_dict,
    lora_state_dict,
    mark_only_lora_trainable,
)
from g2c.notebook_extras.sft import (
    chat_sample,
    plot_sft_history,
    show_base_vs_sft,
    train_sft_with_progress,
)
from g2c.sft import ChatTemplate, SFTTrainer

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

Run the LoRA tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 13B scaffolds (`LoRALinear.forward`, `merge`, `unmerge`, and `mark_only_lora_trainable`).

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_lora.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 13B LoRA tests are not passing yet."

## Load BaseLM twice

This module runs on BaseLM only. LoRA's whole reason to exist is a model too expensive to fully fine-tune, and your course models are cheap to fine-tune by design — `g2c/lora` therefore targets `torch.nn.Linear` trees like BaseLM's (the lesson's scope notes cover this).

We load the artifact twice: `base_artifact` stays pristine for before/after comparisons, `lora_artifact` is the copy we inject and train.

In [ ]:
TRAIN_DEVICE = "auto"
SEED = 13

base_artifact = load_model_artifact_with_tokenizer(
    "BaseLM", repo_root=repo_root, device=TRAIN_DEVICE
)
lora_artifact = load_model_artifact_with_tokenizer(
    "BaseLM", repo_root=repo_root, device=TRAIN_DEVICE
)

base_model = base_artifact.model
lora_model = lora_artifact.model
tokenizer = lora_artifact.tokenizer
template = ChatTemplate()
pad_id = tokenizer.special_to_id.get("<|pad|>", getattr(tokenizer, "pad_token_id", None) or 0)
end_id = tokenizer.special_to_id.get(template.END, getattr(tokenizer, "eos_token_id", None))

hf_config = lora_model.inner.config
print("loaded:", lora_artifact.name)
print("hidden size:", hf_config.hidden_size)
print("layers:", hf_config.num_hidden_layers)
print("attention heads:", hf_config.num_attention_heads, "| kv heads:", hf_config.num_key_value_heads)
print("total parameters:", f"{count_parameters(lora_model)[1]:,}")
print("pad id:", pad_id, "end id:", end_id)

## Exercise 1 - Count before you build

Before touching any code, derive the trainable-parameter count on paper. Rank-8 adapters on `q_proj` and `v_proj`: each adapted layer gets `A (in, r)` plus `B (r, out)`, so `r * (in + out)` parameters. Watch the GQA wrinkle printed above — `v_proj` does not map 960 to 960.

In [ ]:
"Question: With rank 8 on q_proj and v_proj across all 32 layers, how many trainable parameters does LoRA add? Show the per-layer arithmetic (q_proj is 960 to 960; v_proj is 960 to 320 because 5 of 15 heads carry KV). What percentage of the ~362M total is that?"
"Answer: "

## Inject the adapters

The paper's classic recipe: adapt the attention query and value projections, leave everything else alone. `inject_lora` walks the tree and wraps every matching `torch.nn.Linear` in place.

In [ ]:
LORA_RANK = 8
LORA_ALPHA = 8  # scaling = alpha / rank = 1.0

replaced = inject_lora(
    lora_model, {"q_proj", "v_proj"}, rank=LORA_RANK, alpha=LORA_ALPHA
)
print(f"replaced {len(replaced)} layers:")
for name in replaced[:3]:
    print("  ", name)
print("   ...")
for name in replaced[-2:]:
    print("  ", name)

trainable, total = count_parameters(lora_model)
print(f"\nparameters: {trainable:,} trainable / {total:,} total")
print("(nothing is frozen yet -- that is the next step, not inject_lora's job)")

## Exercise 2 - The exact no-op

`B` starts at zero, so every adapter's delta is exactly zero: the injected model must produce **bit-identical** logits to the pristine copy. Not close — identical. If you see `1e-7` here, something initialized `B` wrong.

In [ ]:
probe = "What is the capital of France?"
probe_ids = torch.tensor(
    [tokenizer.encode_with_vocab_size(probe, lora_model.vocab_size)],
    device=lora_model.device,
)
with torch.no_grad():
    base_logits = base_model(probe_ids.to(base_model.device))
    lora_logits = lora_model(probe_ids)

diff = (base_logits.cpu() - lora_logits.cpu()).abs().max().item()
print("max |base - injected| logit difference:", diff)
assert diff == 0.0, "the freshly injected adapter must be an exact no-op"

In [ ]:
"Question: Why must B (and not A) start at zero -- and why not both? Work the chain rule: with B = 0, what is A's gradient on the very first backward pass, and what unfreezes it?"
"Answer: "

## Freeze - and count what the optimizer will see

`mark_only_lora_trainable` is the step that buys the memory. AdamW keeps two state tensors (`m` and `v`) per *trainable* parameter — count the bytes both ways.

In [ ]:
trainable, total = mark_only_lora_trainable(lora_model)
print(f"trainable: {trainable:,} of {total:,} ({100 * trainable / total:.3f}%)")

BYTES_PER_FLOAT32 = 4
full_state = 2 * total * BYTES_PER_FLOAT32
lora_state = 2 * trainable * BYTES_PER_FLOAT32
print(f"\nAdamW state (m + v, float32):")
print(f"  full fine-tuning: {full_state / 1e9:.2f} GB")
print(f"  LoRA:             {lora_state / 1e6:.2f} MB")
print(f"  ratio:            {full_state / lora_state:,.0f}x")

In [ ]:
"Question: Fine-tuning memory has four main tenants: weights, activations, gradients, and optimizer state. Which of the four does LoRA shrink, and which does it leave essentially unchanged? Given that, why does LoRA barely change the wall-clock time per step?"
"Answer: "

## Load your Module 13 dataset

The point of this module is that *only the parameterization changes* — so we reuse your hand-authored instruction pairs from Module 13, byte for byte.

In [ ]:
dataset_path = repo_root / "data" / "work" / "module13" / "instructions.json"
if not dataset_path.exists():
    raise FileNotFoundError(
        f"{dataset_path} not found. Run Module 13's dataset cells first "
        "(./notebook.sh 13) -- Module 13B deliberately reuses that dataset."
    )
sft_pairs = json.loads(dataset_path.read_text())
print(f"loaded {len(sft_pairs)} pairs from {dataset_path.relative_to(repo_root)}")


def messages_from_pair(pair: dict[str, str]) -> list[dict[str, str]]:
    return [
        {"role": "user", "content": pair["user"]},
        {"role": "assistant", "content": pair["assistant"]},
    ]


encoded_examples = [
    template.render_with_mask(
        messages_from_pair(pair),
        tokenizer,
        vocab_size=lora_model.vocab_size,
    )
    for pair in sft_pairs
]

DATA_SPLIT_SEED = 13
perm = torch.randperm(
    len(encoded_examples), generator=torch.Generator().manual_seed(DATA_SPLIT_SEED)
).tolist()
val_count = max(1, len(encoded_examples) // 5)
val_indices = set(perm[:val_count])
train_examples = [ex for i, ex in enumerate(encoded_examples) if i not in val_indices]
val_examples = [ex for i, ex in enumerate(encoded_examples) if i in val_indices]
print("train examples:", len(train_examples))
print("val examples:", len(val_examples))

## Exercise 3 - Train the adapter

Same trainer, same loss, same data as Module 13. The one delta: the model handed to `SFTTrainer` is `LoRAModel(lora_model)` — the course-`Module` view whose `parameters()` exposes only the adapters, so the optimizer state is sized to what you counted above. LoRA runs tolerate (and usually want) a somewhat higher learning rate than full fine-tuning; `1e-3` is a solid starting point here where full SFT used `3e-4`.

In [ ]:
SFT_CONFIG = {
    "max_seq_len": min(128, lora_model.max_seq_len),
    "pad_id": pad_id,
    "batch_size": 4,
    "max_steps": 500,
    "max_lr": 1e-3,
    "min_lr": 1e-4,
    "warmup_steps": 20,
    "weight_decay": 0.01,
    "grad_clip": 1.0,
    "eval_every": 50,
    "eval_iters": 10,
    "log_every": 10,
    "device": TRAIN_DEVICE,
}

trainer = SFTTrainer(
    LoRAModel(lora_model),
    examples=train_examples,
    generator=torch.Generator().manual_seed(SEED),
    **SFT_CONFIG,
)
print("parameters the optimizer sees:", sum(p.numel() for p in trainer.model.parameters()))

history = train_sft_with_progress(
    "BaseLM LoRA SFT", trainer, eval_examples=val_examples
)
plot_sft_history(history)

## Compare: base vs LoRA-tuned

`lora_artifact.model` is the injected model — the adapters are live inside it, so the Module 13 sampling helpers work unchanged.

In [ ]:
heldout_prompts = [
    "What is the capital of France?",
    "Give one tip for training small models.",
    "What does the loss mask do in SFT?",
]
ood_prompts = [
    "Write two sentences about the ocean.",
    "Continue: To be, or not to be,",
]
show_base_vs_sft(base_artifact, lora_artifact, heldout_prompts, seed=SEED)
show_base_vs_sft(base_artifact, lora_artifact, ood_prompts, seed=SEED)

In [ ]:
"Question: Put these samples next to your Module 13 full-SFT samples (same data, same step count). Where do the two land in the same place, and where do they differ -- format compliance, stopping on <|end|>, verbatim memorization, damage to base behavior on the prose continuation?"
"Answer: "

## Exercise 4 - Merge for deployment

Serving with the unmerged adapter pays two extra skinny matmuls per adapted layer on every forward. `merge()` folds `A @ B` into the frozen weight so inference costs exactly one linear layer — then verify the function didn't move.

In [ ]:
from g2c.lora import LoRALinear


def median_forward_seconds(model, ids, *, repeats: int = 10) -> float:
    times = []
    with torch.no_grad():
        for _ in range(repeats):
            if torch.backends.mps.is_available():
                torch.mps.synchronize()
            start = time.perf_counter()
            model(ids)
            if torch.backends.mps.is_available():
                torch.mps.synchronize()
            times.append(time.perf_counter() - start)
    return sorted(times)[len(times) // 2]


with torch.no_grad():
    unmerged_logits = lora_model(probe_ids).cpu()
unmerged_time = median_forward_seconds(lora_model, probe_ids)

for module in lora_model.modules():
    if isinstance(module, LoRALinear):
        module.merge()

with torch.no_grad():
    merged_logits = lora_model(probe_ids).cpu()
merged_time = median_forward_seconds(lora_model, probe_ids)
base_time = median_forward_seconds(base_model, probe_ids.to(base_model.device))

print("max |unmerged - merged| logit difference:", (unmerged_logits - merged_logits).abs().max().item())
print(f"forward time  unmerged: {unmerged_time * 1e3:.1f} ms")
print(f"forward time  merged:   {merged_time * 1e3:.1f} ms")
print(f"forward time  base:     {base_time * 1e3:.1f} ms")

In [ ]:
"Question: The unmerged/merged logit difference above is small but (unlike Exercise 2) usually not zero. Both compute 'the same' function -- where does the difference come from?"
"Answer: "

## Exercise 5 - Unmerge: the eject button

Subtract the delta back out and compare against the pristine copy. This is the guarantee full fine-tuning cannot offer: keep the base weights untouched and the behavior change lives entirely in a removable attachment.

In [ ]:
for module in lora_model.modules():
    if isinstance(module, LoRALinear):
        module.unmerge()

pristine = dict(base_model.named_parameters())
worst = 0.0
for name, p in lora_model.named_parameters():
    if name.endswith("lora_A") or name.endswith("lora_B"):
        continue
    clean_name = name.replace(".base.weight", ".weight").replace(".base.bias", ".bias")
    worst = max(worst, (p.detach().cpu() - pristine[clean_name].detach().cpu()).abs().max().item())
print("max |unmerged base weight - pristine weight|:", worst)

In [ ]:
"Question: After a merge/unmerge round trip the base weights are close to pristine but not bit-identical. Why? And why does the recommended workflow -- never merge the training copy; keep the base artifact pristine and the adapter in its own file -- make 'zero forgetting' true by construction rather than by hope?"
"Answer: "

## Exercise 6 - Ship the adapter, not the model

The durable output of a LoRA run is the A/B matrices. Save them, then prove the point: load a *fresh* BaseLM, inject blank adapters, drop the file in, and the behavior arrives with it.

(This cell briefly holds a third copy of BaseLM in memory — see the lesson's M-series notes if you're on 8 GB.)

In [ ]:
adapter_dir = repo_root / "data" / "work" / "module13b"
adapter_dir.mkdir(parents=True, exist_ok=True)
adapter_path = adapter_dir / "lora-adapter.pt"
torch.save(lora_state_dict(lora_model), adapter_path)

adapter_mb = adapter_path.stat().st_size / 1e6
model_mb = count_parameters(base_model)[1] * 4 / 1e6  # float32 weights
print(f"adapter file: {adapter_mb:.1f} MB")
print(f"base weights: {model_mb:,.0f} MB")
print(f"ratio: {model_mb / adapter_mb:,.0f}x smaller")

fresh_artifact = load_model_artifact_with_tokenizer(
    "BaseLM", repo_root=repo_root, device=TRAIN_DEVICE
)
inject_lora(fresh_artifact.model, {"q_proj", "v_proj"}, rank=LORA_RANK, alpha=LORA_ALPHA)
load_lora_state_dict(fresh_artifact.model, torch.load(adapter_path))
print("\nfresh BaseLM + adapter file:")
print(chat_sample(fresh_artifact, "What is the capital of France?", seed=SEED))

In [ ]:
"Question: What exactly must match between the saving side and the loading side for an adapter file to work -- and what happens (silently or loudly) if the base model's weights are different, e.g. a later revision of the same model?"
"Answer: "

## Exercise 7 (optional) - Rank sweep

Is rank 8 doing anything rank 1 can't? Each run below is a fresh injection trained for fewer steps, so this stays a coffee-break experiment. Add ranks to taste.

In [ ]:
sweep_results = {}
for rank in (1, 8):
    sweep_artifact = load_model_artifact_with_tokenizer(
        "BaseLM", repo_root=repo_root, device=TRAIN_DEVICE
    )
    inject_lora(sweep_artifact.model, {"q_proj", "v_proj"}, rank=rank, alpha=rank)
    n_trainable, _ = mark_only_lora_trainable(sweep_artifact.model)
    sweep_trainer = SFTTrainer(
        LoRAModel(sweep_artifact.model),
        examples=train_examples,
        generator=torch.Generator().manual_seed(SEED),
        **{**SFT_CONFIG, "max_steps": 150, "eval_every": 150},
    )
    sweep_history = train_sft_with_progress(
        f"LoRA rank {rank}", sweep_trainer, eval_examples=val_examples
    )
    sweep_results[rank] = {
        "trainable": n_trainable,
        "final_val": sweep_history["val_loss"][-1] if sweep_history["val_loss"] else None,
        "sample": chat_sample(sweep_artifact, "What is the capital of France?", seed=SEED),
    }
    del sweep_artifact, sweep_trainer

for rank, r in sweep_results.items():
    print(f"rank {rank}: {r['trainable']:,} params, final val loss {r['final_val']}")
    print("  sample:", r["sample"][:120].replace("\n", " "))

In [ ]:
"Question: How much did rank matter for this task, in val loss and in the samples? Module 13 taught that SFT mostly teaches a format. What does your sweep say about how many directions of weight-change that format shift actually needs?"
"Answer: "

## Final notes

The recipe you just ran is, at production scale, the recipe: QLoRA is this plus a quantized frozen base; every "fine-tune Llama on your laptop" tutorial is this with a bigger model. You now know exactly which matrices move and why the rest can't.

When complete, ask a coding agent to grade your Module 13B notebook. Partial work is fine: the agent should grade what's answered and skip blank answers.

In [ ]:
"Question: In one paragraph: what does LoRA change about fine-tuning, and what does it leave exactly the same? Name the one flag all the memory savings trace back to."
"Answer: "